<!-- NVN-HEADER -->
# Trial-level verification — is the amplitude signal genuine?

**Status:** the methodologically correct version of the amplitude result.

## Purpose
Determine whether the cross-subject amplitude signal is genuine **trial-level** prediction, or
merely **subject-ranking** (some people simply respond more strongly on average).

## What we did
Removed the subject mean before correlating, and ran a proper leave-one-subject-out protocol
alongside the original single split.

## What we achieved
- **fNIRS trial-level r = 0.33 (p < 0.001)** — genuine, and the project's strongest positive.
- fMRI trial-level r = 0.08 — real but weak.
- Showed the naive single-split 0.21 collapses to ~0.01 under LOSO — an important
  methodological correction.

## What we failed to achieve
- Did not establish cross-subject **waveform** recovery (not its aim).
- The fMRI amplitude signal remained weak.

> **Retrospective (data-quality audit, 2026-07):** this notebook trained against the **HEFMI** fNIRS target, later shown to be **systemic-dominated** (HbO/HbR = **+0.898**, should be negative) with split-half reliability of only **0.127** after the pipeline's bandpass -> global-signal regression -> z-score chain. GSR alone removes ~83% of the evoked response. **The negative results below are confounded: they measure the preprocessing, not EEG.** See `NeuroVascNet_Data_Quality_Audit.md` sections 2.0-2.0c.

# NeuroVascNet - Cross-Subject Amplitude: TRIAL-LEVEL vs SUBJECT-RANKING Verification

## What this test is for
We found EEG can predict the hemodynamic response **amplitude** across subjects
(fNIRS r=0.25 with CORAL, fMRI r=0.15 plain). But a cross-subject correlation can be
**two very different things**, and we must know which:

- **TRIAL-LEVEL (strong claim):** within one person, EEG predicts *which windows have a
  bigger response than others*. This is genuine, useful prediction.
- **SUBJECT-RANKING (weak claim):** EEG only predicts that *subject A is generally more
  responsive than subject B*. The pooled correlation looks high, but it is just ranking
  people by their average responsiveness - not predicting anything within a person.

The plain (global) null cannot tell these apart, because shuffling across all pooled
windows destroys BOTH kinds of structure. This notebook separates them.

## How it separates them (3 tests)
1. **Global null** - shuffle amplitudes across ALL pooled windows. Beating it = "some
   cross-subject signal exists" (either kind).
2. **Within-subject null** - shuffle amplitudes ONLY inside each subject (keeps each
   subject's mean, destroys trial order). If the real result still beats this, the signal
   is not merely subject-ranking.
3. **Subject-mean-removed correlation (the decisive one)** - subtract each subject's own
   mean amplitude from both prediction and truth, then correlate. This measures **pure
   within-subject trial-level prediction** with all between-subject differences removed.
   If this is > 0 and beats its null -> **TRIAL-LEVEL signal is real.**

## Method
- EEG features = band-power (mean power per electrode per band, 128-d).
- Target = response amplitude per window = std over time of the RAW hemo (amplitude-preserving).
- Prediction = Ridge, leave-one-subject-out (LOSO). fNIRS uses **CORAL** domain adaptation
  (aligns each training subject's features to the held-out subject; uses target EEG only,
  never target labels = valid unsupervised adaptation). fMRI uses plain (CORAL hurts it).

## RESULT (this run)
| modality | cross-subj r | trial-level r (subj-mean removed) | p (trial) | verdict |
|---|---|---|---|---|
| **fNIRS (CORAL)** | +0.253 | **+0.330** | <0.001 | **TRIAL-LEVEL (strong)** |
| **fMRI (plain)**  | +0.148 | +0.083 | 0.037 | TRIAL-LEVEL (weaker but real) |

**Conclusion:** the fNIRS result is genuine **within-subject** trial-level prediction (its
subject-mean-removed r of 0.33 is even *higher* than the pooled 0.25) - not a subject-ranking
artifact. fMRI also shows real trial-level signal, but weaker. Both are honest positives.

In [ ]:
# CELL 1  helpers: load, band-power features, CORAL
import glob, numpy as np
from scipy.signal import butter, sosfiltfilt, hilbert
from sklearn.linear_model import Ridge

def load(pat):
    f=sorted(glob.glob('neurovascnet_outputs/**/'+pat+'.npz',recursive=True))[0]
    z=np.load(f); return z['eeg'],z['hemo'],z['subject_ids']
BANDS=[(4,8),(8,13),(13,30),(30,45)]
def bpf(eeg,fs=200,chunk=256):
    sos=[butter(4,[lo,hi],btype='band',fs=fs,output='sos') for lo,hi in BANDS]; o=[]
    for i in range(0,len(eeg),chunk):
        e=eeg[i:i+chunk]; f=[np.abs(hilbert(sosfiltfilt(s,e,axis=-1),axis=-1)).mean(-1) for s in sos]
        o.append(np.concatenate(f,1))
    return np.concatenate(o,0).astype(np.float32)
def _sq(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V*np.sqrt(w))@V.T
def _isq(C):
    w,V=np.linalg.eigh(C); w=np.clip(w,1e-10,None); return (V/np.sqrt(w))@V.T
def coral(Xs,Xt,eps=1e-1):
    ms,mt=Xs.mean(0),Xt.mean(0)
    Cs=np.cov(Xs-ms,rowvar=False)+eps*np.eye(Xs.shape[1]); Ct=np.cov(Xt-mt,rowvar=False)+eps*np.eye(Xs.shape[1])
    return (Xs-ms)@(_isq(Cs)@_sq(Ct))+mt

In [ ]:
# CELL 2  LOSO predictions (pooled), then the 3-way trial-level analysis
def loso_preds(X,Y,sid,use_coral,alpha=10.):
    P=[];T=[];S=[]
    for s in np.unique(sid):
        te=sid==s; tr=~te
        if te.sum()<3: continue
        Xtr,Xte=X[tr].copy(),X[te].copy()
        if use_coral: Xtr=coral(Xtr,Xte)
        mu,sd=Xtr.mean(0),Xtr.std(0)+1e-6; ym,ys=Y[tr].mean(0),Y[tr].std(0)+1e-6
        p=Ridge(alpha=alpha).fit((Xtr-mu)/sd,(Y[tr]-ym)/ys).predict((Xte-mu)/sd)
        P.append(p.reshape(te.sum(),-1)); T.append(((Y[te]-ym)/ys).reshape(te.sum(),-1)); S.append(sid[te])
    return np.concatenate(P),np.concatenate(T),np.concatenate(S)

def colcorr(P,Y):
    Pc=P-P.mean(0);Yc=Y-Y.mean(0);pv=(Pc*Pc).sum(0);yv=(Yc*Yc).sum(0);ok=(pv>1e-8)&(yv>1e-8)
    r=np.zeros(Y.shape[1]);r[ok]=(Pc*Yc).sum(0)[ok]/np.sqrt(pv[ok]*yv[ok]);return r,ok
def center_by_subject(A,S):
    A=A.copy()
    for s in np.unique(S): m=S==s; A[m]=A[m]-A[m].mean(0)
    return A
def shuffle_within(T,S,rng):
    T=T.copy()
    for s in np.unique(S): m=np.where(S==s)[0]; T[m]=T[m][rng.permutation(len(m))]
    return T

def analyze(P,T,S,n_null=300,seed=0):
    rng=np.random.default_rng(seed); ok=colcorr(P,T)[1]
    r_glob=colcorr(P,T)[0][ok].mean()
    ng=np.array([colcorr(P,T[rng.permutation(len(T))])[0][ok].mean() for _ in range(n_null)])
    nw=np.array([colcorr(P,shuffle_within(T,S,rng))[0][ok].mean() for _ in range(n_null)])
    Pc,Tc=center_by_subject(P,S),center_by_subject(T,S); ok2=colcorr(Pc,Tc)[1]
    r_trial=colcorr(Pc,Tc)[0][ok2].mean()
    nt=np.array([colcorr(Pc,shuffle_within(Tc,S,rng))[0][ok2].mean() for _ in range(n_null)])
    return dict(r_glob=float(r_glob),p_glob=float((ng>=r_glob).mean()),
                p_within=float((nw>=r_glob).mean()),
                r_trial=float(r_trial),p_trial=float((nt>=r_trial).mean()))

In [ ]:
# CELL 3  RUN: is the cross-subject amplitude TRIAL-LEVEL or just SUBJECT-RANKING?
Ef,Hf,Sf=load('fnirs_train_w30_e32*'); Em,Hm,Sm=load('ds002336_train_w30_e32*')
Xf,Xm=bpf(Ef),bpf(Em)
for nm,X,H,sid,use_coral in [('fNIRS (CORAL)',Xf,Hf,Sf,True),('fMRI (plain)',Xm,Hm,Sm,False)]:
    Y=H.std((1,2)).reshape(-1,1).astype(np.float32)
    P,T,S=loso_preds(X,Y,sid,use_coral); a=analyze(P,T,S)
    v='TRIAL-LEVEL (strong)' if (a['p_trial']<0.05 and a['r_trial']>0.02) else 'mostly SUBJECT-RANKING'
    print('=== %s overall amplitude ==='%nm)
    print('  cross-subject r          = %+.4f  (global null p=%.3f)'%(a['r_glob'],a['p_glob']))
    print('  beats WITHIN-subject null: p=%.3f'%a['p_within'])
    print('  TRIAL-LEVEL r (subj-mean removed) = %+.4f  (p=%.3f)'%(a['r_trial'],a['p_trial']))
    print('  -> %s\n'%v)